In [1]:
# Example Data set

import pandas as pd
import numpy as np

data = {
    "CustomerID": [101, 102, 103, 104, 104, 105],
    "Name": ["Alice", "Bob", "Charlie", "David", "David", "Eve"],
    "Age": [25, 30, np.nan, 45, 45, -5],
    "Income": [50000, 60000, 55000, 70000, 70000, "unknown"],
    "Gender": ["Female", "Male", "male", "MALE", "MALE", "Female"],
    "Joined_Date": ["2023-01-10", "2023-02-15", "2023-03-20",
                    "2023-03-20", "2023-03-20", "not_a_date"],
    "City": ["Mumbai", "Delhi", "Mumbai", "Delhi", "Delhi", "Bangalore"]
}

df = pd.DataFrame(data)

df

,CustomerID,Name,Age,Income,Gender,Joined_Date,City
0,101,Alice,25.0,50000,Female,2023-01-10,Mumbai
1,102,Bob,30.0,60000,Male,2023-02-15,Delhi
2,103,Charlie,NaN,55000,male,2023-03-20,Mumbai
3,104,David,45.0,70000,MALE,2023-03-20,Delhi
4,104,David,45.0,70000,MALE,2023-03-20,Delhi
5,105,Eve,-5.0,unknown,Female,not_a_date,Bangalore


### Level 1 — Column-Level Cleaning

(Structural fixes)

These operations change the **structure or metadata** of the DataFrame.

### Things you do on columns:

1. **Rename columns**

   * Fix messy names
   * Remove spaces
   * Standardize naming

2. **Drop useless columns**

   * IDs
   * Leakage columns
   * Constant columns

3. **Change data types**

   * `"12"` → 12
   * object → float
   * int → category
   * string → datetime

4. **Handle duplicate columns (rare but possible)**

These operations affect how the dataset is structured.

---

### Level 2 — Data-Level Cleaning

(Content fixes)

These operations change the **values inside columns**.

### Things you do on data:

1. **Handle missing values**

   * Drop rows
   * Fill with mean/median/mode
   * Custom logic

2. **Remove duplicate rows**

3. **Fix invalid values**

   * Negative ages
   * Impossible categories
   * Out-of-range values

4. **Normalize text categories**

   * "Male", "male", "MALE" → "male"

---



In [2]:
# To know the problems column level problems do this
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CustomerID   6 non-null      int64  
 1   Name         6 non-null      object 
 2   Age          5 non-null      float64
 3   Income       6 non-null      object 
 4   Gender       6 non-null      object 
 5   Joined_Date  6 non-null      object 
 6   City         6 non-null      object 
dtypes: float64(1), int64(1), object(5)
memory usage: 464.0+ bytes


#### Column Level Problem : Renaming Column

### What is Renaming Columns?

It means changing column labels without touching the data itself.

You are NOT changing values.
You are changing the **metadata** (structure).

Example:

```
"Customer Name"   →   "customer_name"
"AGE (Years)"     →   "age"
"Total$Price"     →   "total_price"
```

Cleaner names = faster coding + fewer bugs.

---

### Why Do We Rename Columns?

Brutal truth: most beginners skip this and suffer later.

You rename columns when:

✔️ Column names have spaces
✔️ Upper/lowercase is inconsistent
✔️ Special characters exist (`$ % ( ) -`)
✔️ Names are too long
✔️ You want ML-friendly naming

Because this:

```python
df["Customer Name"]
```

is annoying and error-prone compared to:

```python
df.customer_name
```

---

### Main Syntax 

### Rename specific columns

```
df.rename(columns={"old_name": "new_name"})
```

Example:

```python
import pandas as pd

df = pd.DataFrame({
    "Customer Name": ["A", "B"],
    "AGE": [22, 30]
})

df = df.rename(columns={
    "Customer Name": "customer_name",
    "AGE": "age"
})
```

Result:

```
customer_name   age
A               22
B               30
```

👉 This is the safest and most common way.

---

### Rename ALL columns at once

If you want full control:

```
df.columns = ["col1", "col2", "col3"]
```

Example:

```python
df.columns = ["name", "age"]
```

---

### Automatic cleaning 

You’ll do this a LOT:

```python
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)
```

Transforms:

```
"Customer Name" → "customer_name"
"Total Price"   → "total_price"
```

This is professional-level preprocessing.

---

### When is it used in real?

Almost immediately after loading data:

```python
df = pd.read_csv("data.csv")

# Step 1: clean column names
df.columns = df.columns.str.lower().str.replace(" ", "_")
```


In [3]:
# get the columns
print("Before : ", df.columns)

print("After : ",df.columns.str.lower().str.replace(" ","_"))
df.columns = df.columns.str.lower().str.replace(" ","_")
# the str is an accessor that exposes a set of vectorized string methods to be applied to the elements of a Series or Index

Before :  Index(['CustomerID', 'Name', 'Age', 'Income', 'Gender', 'Joined_Date', 'City'], dtype='object')
After :  Index(['customerid', 'name', 'age', 'income', 'gender', 'joined_date', 'city'], dtype='object')


### Column Level Problem : Dropping Useless Columns

#### What does “dropping useless columns” mean?

It means removing columns that don’t help your analysis or model.

You’re changing the **structure** of the DataFrame.

You are basically saying:

👉 *“This feature adds zero value — get it out.”*

---

####  When do we drop columns?

Be brutally practical here. You drop columns when they are:

#### 1. Pure IDs

Example:

```
user_id
transaction_id
serial_number
```

These are unique identifiers, not patterns.
ML models can’t learn from randomness.

---

#### Leakage columns 

Columns that reveal the answer.

Example:

```
approved_loan
final_score
prediction_result
```

---

#### 3. Constant columns

```
country = "India" for all rows
```

Zero variation = zero information.

---

#### 4. Too many missing values

If 80–90% is NaN, sometimes it’s just junk.

---

#### 5. Redundant columns

Example:

```
full_name
first_name
last_name
```

Sometimes one is enough.

---

#### Syntax — How to drop columns in pandas

#### Drop one column

```python
df = df.drop(columns=["user_id"])
```

or

```python
df.drop("user_id", axis=1, inplace=True)
```

👉 `axis=1` means column.

---

#### Drop multiple columns

```python
df = df.drop(columns=["user_id", "timestamp", "temp_col"])
```

This is the most common real-world usage.

---

In [4]:
print("before", df.columns)
print("After", df.drop(columns=["age"])) 

before Index(['customerid', 'name', 'age', 'income', 'gender', 'joined_date', 'city'], dtype='object')
After    customerid     name   income  gender joined_date       city
0         101    Alice    50000  Female  2023-01-10     Mumbai
1         102      Bob    60000    Male  2023-02-15      Delhi
2         103  Charlie    55000    male  2023-03-20     Mumbai
3         104    David    70000    MALE  2023-03-20      Delhi
4         104    David    70000    MALE  2023-03-20      Delhi
5         105      Eve  unknown  Female  not_a_date  Bangalore


#### Column Level Problem : Changing the data type of the column
#### What data types can a pandas column have?

These are the ones you’ll actually see in real datasets:

####  Numeric types

* `int64` → integers
* `float64` → decimals

Used for: age, price, salary, scores.

---

####  Text type

* `object` → old generic string type
* `string` → newer dedicated string dtype

Most CSV text columns load as `object`.

---

#### Boolean

* `bool` → True / False

---

#### Datetime

* `datetime64[ns]`

Used for timestamps, signup dates, logs.

---

#### Category (VERY IMPORTANT for ML)

* `category`

Stores limited repeating labels efficiently.

Example:

```
male, female, other
```

---

#### Preferred syntax for converting dtype

There are many ways — but honestly, you should mostly use
**two functions** in real work.

---

#### 1) `astype()` — General purpose (MOST COMMON)

```python
df["age"] = df["age"].astype("int64")
```

Examples:

```python
df["price"] = df["price"].astype("float64")
df["gender"] = df["gender"].astype("category")
df["flag"] = df["flag"].astype("bool")
```

This is the most standard syntax.

---

#### 2) `pd.to_numeric()` / `pd.to_datetime()` — SAFER conversion

Used when data is messy.

```python
df["price"] = pd.to_numeric(df["price"])
df["date"]  = pd.to_datetime(df["date"])
```

Real datasets often need these instead of `astype()`.

---

#### Internal conversion (what actually happens)

When you convert:

```python
df["age"] = df["age"].astype("int64")
```

pandas:

1. Reads every value in the column
2. Tries to parse it into target type
3. Allocates new memory
4. Rebuilds the column

It is NOT just a label change.
It actually rebuilds the array internally.

---

#### What happens to invalid values during conversion?

This is where beginners get wrecked.

#### Case 1 — Using `astype()`

Example column:

```
["10", "20", "abc"]
```

Code:

```python
df["col"].astype("int64")
```

👉 Result:

```
ValueError: invalid literal
```

`astype()` is strict.
It crashes if anything is invalid.

---

#### Case 2 — Using safer conversion 

```python
df["col"] = pd.to_numeric(df["col"], errors="coerce")
```

Now:

```
"abc" → NaN
```

Invalid values become **NaN**.

Same idea for dates:

```python
df["date"] = pd.to_datetime(df["date"], errors="coerce")
```

Invalid dates → `NaT` (Not-a-Time).

---

#### Professional rule 

Use:

```
astype()         → when data is already clean
to_numeric()     → when data is messy
to_datetime()    → always for dates
```

---

#### Honest insight most tutorials won’t tell you

90% of “data cleaning” in pandas is actually:

```
object  → numeric
object  → datetime
object  → category
```

CSV imports almost always load messy columns as `object`.

---


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customerid   6 non-null      int64  
 1   name         6 non-null      object 
 2   age          5 non-null      float64
 3   income       6 non-null      object 
 4   gender       6 non-null      object 
 5   joined_date  6 non-null      object 
 6   city         6 non-null      object 
dtypes: float64(1), int64(1), object(5)
memory usage: 464.0+ bytes


In [6]:
# Changing income from object to numeric
df["income"] = pd.to_numeric(df["income"], errors="coerce")

# Changing the joined_date to date time
df["joined_date"] = pd.to_datetime(df["joined_date"], errors="coerce")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   customerid   6 non-null      int64         
 1   name         6 non-null      object        
 2   age          5 non-null      float64       
 3   income       5 non-null      float64       
 4   gender       6 non-null      object        
 5   joined_date  5 non-null      datetime64[ns]
 6   city         6 non-null      object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 464.0+ bytes


### Data level Handling : Handling Missing Values

#### What are Missing Values in pandas?

Missing values mean **no data exists** for that cell.

In pandas they usually appear as:

```
NaN   → numeric missing
None  → Python null
NaT   → datetime missing
```

You’ll see them after:

* CSV imports
* Bad conversions
* Manual data entry
* `errors="coerce"` conversions

---

#### Step 1 — Checking Missing Values

Before fixing anything, always measure the damage.

#### Check total missing values per column

```python
df.isna().sum()
```

Example output:

```
age        5
salary     0
gender     2
```

This tells you where the real problem is.

---

#### Check percentage (VERY IMPORTANT habit)

Counts alone don’t mean much.

```python
df.isna().mean() * 100
```

Now you know if a column is 2% missing or 80% missing —
huge difference in strategy.

---

#### See rows containing missing values

```python
df[df.isna().any(axis=1)]
```

Useful for debugging messy rows.

---

#### Step 2 — Handling Missing Values

There is NO single rule.
You choose based on meaning of the column.

---

#### Option 1 — Drop missing values

When data is small or missing rows are useless.

### Drop rows with missing values

```python
df = df.dropna()
```

### Drop only if specific column is missing

```python
df = df.dropna(subset=["age"])
```

---

#### Brutal truth:

Blindly dropping rows is the #1 beginner mistake.
You might accidentally delete half your dataset.

Always check counts first.

---

#### Option 2 — Fill missing values (Most common)

#### Fill with mean (numeric columns)

```python
df["age"] = df["age"].fillna(df["age"].mean())
```

Used when values are continuous.

---

#### Fill with median

```python
df["salary"] = df["salary"].fillna(df["salary"].median())
```

Better when data has outliers.

---

#### Fill with mode (categorical)

```python
df["gender"] = df["gender"].fillna(df["gender"].mode()[0])
```

---

#### Fill with constant value

```python
df["city"] = df["city"].fillna("unknown")
```

Very common in real pipelines.

---

#### Option 3 — Forward / Backward Fill (time series)

```python
df["price"] = df["price"].ffill()
df["price"] = df["price"].bfill()
```

Used when rows are ordered.

---

#### Real Workflow 

1️. Inspect:

```python
df.isna().sum()
```

2️. Decide strategy per column:

```
numeric → mean/median
category → mode or "unknown"
datetime → ffill/bfill or drop
```

3️. Apply fill.

---


In [9]:
df

,customerid,name,age,income,gender,joined_date,city
0,101,Alice,25.0,50000.0,Female,2023-01-10,Mumbai
1,102,Bob,30.0,60000.0,Male,2023-02-15,Delhi
2,103,Charlie,NaN,55000.0,male,2023-03-20,Mumbai
3,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
4,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
5,105,Eve,-5.0,NaN,Female,NaT,Bangalore


In [12]:
# Checking the presence of missing values

# Sum of missing values per column
df.isna().sum()

customerid     0
name           0
age            1
income         1
gender         0
joined_date    1
city           0
dtype: int64

In [13]:
# Checking missing values in terms of percentage

df.isna().mean() * 100

customerid      0.000000
name            0.000000
age            16.666667
income         16.666667
gender          0.000000
joined_date    16.666667
city            0.000000
dtype: float64

In [15]:
# Missing values checking row wise

mask = df.isna().any(axis = 1) # returns a mask 

df[mask]

,customerid,name,age,income,gender,joined_date,city
2,103,Charlie,NaN,55000.0,male,2023-03-20,Mumbai
5,105,Eve,-5.0,NaN,Female,NaT,Bangalore


In [16]:
# Handling missing values

# Option 1 : Dropping Missing Values

copy = df.copy()
copy

,customerid,name,age,income,gender,joined_date,city
0,101,Alice,25.0,50000.0,Female,2023-01-10,Mumbai
1,102,Bob,30.0,60000.0,Male,2023-02-15,Delhi
2,103,Charlie,NaN,55000.0,male,2023-03-20,Mumbai
3,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
4,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
5,105,Eve,-5.0,NaN,Female,NaT,Bangalore


In [ ]:
copy.dropna() # drops the rows which are having missing values

,customerid,name,age,income,gender,joined_date,city
0,101,Alice,25.0,50000.0,Female,2023-01-10,Mumbai
1,102,Bob,30.0,60000.0,Male,2023-02-15,Delhi
3,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
4,104,David,45.0,70000.0,MALE,2023-03-20,Delhi


In [33]:
# Option 2 : Filling the missing values with the most common value

# For numerical fill with mean or median
df["income"] = df["income"].fillna(df["income"].mean())

# Filling the categorical with 

df["joined_date"] = df["joined_date"].ffill()
df

,customerid,name,age,income,gender,joined_date,city
0,101,Alice,25.0,50000.0,Female,2023-01-10,Mumbai
1,102,Bob,30.0,60000.0,Male,2023-02-15,Delhi
2,103,Charlie,NaN,55000.0,male,2023-03-20,Mumbai
3,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
4,104,David,45.0,70000.0,MALE,2023-03-20,Delhi
5,105,Eve,-5.0,61000.0,Female,2023-03-20,Bangalore


#### Data Level Handling : Handling Duplicate Rows and Columns

---

#### Removing Duplicate Rows

#### What is a duplicate row?

When two or more rows have identical values.

Example:

```
name   age
A      20
B      25
A      20   ← duplicate
```

This can happen after merges, bad imports, or logging systems.

---

#### Preferred syntax (most common)

##### Check duplicates first

```python
df.duplicated()
```

Returns True/False per row.

---

### Remove duplicate rows

```python
df = df.drop_duplicates()
```

That’s the clean, standard way.

---

#### Keep specific occurrence

Default keeps **first** row.

```python
df = df.drop_duplicates(keep="first")
```

Other options:

```python
keep="last"
keep=False   # removes ALL duplicates
```

---

#### Remove duplicates based on specific columns

Very common in real pipelines.

Example: user may appear multiple times but same email.

```python
df = df.drop_duplicates(subset=["email"])
```

You’re saying:

👉 “If email repeats, keep only one.”

---

#### mistakes people make

1. Removing duplicates without checking why they exist
Sometimes duplicates are real repeated events.

2. Dropping before sorting
You might keep the wrong version of the row.

Pro move:

```python
df = df.sort_values("date")
df = df.drop_duplicates(subset=["user_id"], keep="last")
```

---

#### Removing Duplicate Columns

Yes, this happens — especially after joins or messy CSVs.

Example:

```
age | salary | age
```

Two columns with same name or same data.

---

#### Detect duplicate column names

```python
df.columns.duplicated()
```

Returns boolean mask.

---

#### Preferred syntax to remove duplicate columns

```python
df = df.loc[:, ~df.columns.duplicated()]
```

This is the clean professional way.

Explanation:

* `:` → all rows
* `~` → NOT
* keeps only unique columns

---

#### If columns have different names but same values

That’s trickier — you compare data:

```python
df.T.drop_duplicates().T
```

Transpose → drop duplicate rows → transpose back.

Very useful trick most beginners don’t know.

---


In [35]:
# Creation of data set

import pandas as pd

data = [
    [1, "A", 25, "2024-01-01", 50000],
    [2, "B", 30, "2024-01-02", 60000],
    [1, "A", 25, "2024-01-01", 50000],  # duplicate row
    [3, "C", 22, "2024-01-03", 45000],
    [2, "B", 30, "2024-01-02", 60000],  # duplicate row
]

df = pd.DataFrame(
    data,
    columns=[
        "user_id",
        "name",
        "age",
        "signup_date",
        "salary"
    ]
)

# Create duplicate column VALUES
df["age_copy"] = df["age"]

# Create duplicate column NAME
df.columns = [
    "user_id",
    "name",
    "age",
    "signup_date",
    "salary",
    "age"     # duplicate column name
]

df

,user_id,name,age,signup_date,salary,age
0,1,A,25,2024-01-01,50000,25
1,2,B,30,2024-01-02,60000,30
2,1,A,25,2024-01-01,50000,25
3,3,C,22,2024-01-03,45000,22
4,2,B,30,2024-01-02,60000,30


In [ ]:
# Check if duplicate rows are present or not
mask = df.duplicated(keep="first")

# Apply mask row wise
df.loc[~mask]

# there is other way to do
df.drop_duplicates() #removes all duplicate rows


,user_id,name,age,signup_date,salary,age
0,1,A,25,2024-01-01,50000,25
1,2,B,30,2024-01-02,60000,30
3,3,C,22,2024-01-03,45000,22


In [48]:
# Checking the presence of the duplicate columns
mask = df.columns.duplicated()

df.loc[:,~mask]

,user_id,name,age,signup_date,salary
0,1,A,25,2024-01-01,50000
1,2,B,30,2024-01-02,60000
2,1,A,25,2024-01-01,50000
3,3,C,22,2024-01-03,45000
4,2,B,30,2024-01-02,60000


### Data Level Handling : Invalid Data

#### What are “invalid values”?

Values that exist in the dataset but **shouldn’t**.

Not missing.
Not duplicate.
Just wrong.

Examples:

```
age = -5           ← impossible
salary = 99999999  ← unrealistic outlier
gender = "mle"     ← typo category
```

Pandas won’t warn you. You must detect them.

---

#### Detecting Invalid Values

Before fixing anything, identify them.

---

#### Negative values

Example:

```python
df[df["age"] < 0]
```

This filters only invalid rows.

---

#### Out-of-range values

Define your logical range.

Example:

```python
df[(df["age"] < 0) | (df["age"] > 100)]
```

You’re telling pandas:

👉 “Give me impossible ages.”

---

#### Impossible categories

First check unique values:

```python
df["gender"].unique()
```

You might see:

```
["Male", "male", "Mle", "Female"]
```

That’s dirty categorical data.

---

#### Step 2 — Fixing Invalid Values

There is no single universal fix.
You choose based on context.

---

#### Method 1 — Replace invalid values with NaN (MOST COMMON)

Move:

👉 Mark them as missing first.
👉 Handle later with normal missing-value logic.

Example:

```python
import numpy as np

df.loc[df["age"] < 0, "age"] = np.nan
```

Why this is good:

* keeps pipeline consistent
* avoids inventing fake numbers

---

#### Method 2 — Clip values into range

Used when values are slightly out of range.

Example:

```python
df["age"] = df["age"].clip(lower=0, upper=100)
```

If age = 150 → becomes 100.

Used in sensor or measurement data.

---

#### Method 3 — Remove invalid rows

If data is totally unreliable.

```python
df = df[df["age"] >= 0]
```

Be careful — dropping rows reduces data size.

---

#### Method 4 — Fix categorical typos

Normalize first:

```python
df["gender"] = df["gender"].str.lower()
```

Then map:

```python
df["gender"] = df["gender"].replace({
    "mle": "male",
    "femle": "female"
})
```

Real-world datasets ALWAYS need this.

---

# Common Pattern 

Most pros follow this workflow:

```
Detect invalid → Convert to NaN → Handle via fill/drop
```

Instead of directly forcing values.

---